<a href="https://colab.research.google.com/github/EgorNezo/-/blob/add-name-Dasha/Variant_12_Garmoniki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
import os
from google.colab import drive
drive.mount('/content/drive')

# ======================================================
# 1. ЗАГРУЗКА ДАННЫХ
# ======================================================
df = pd.read_csv('/content/drive/MyDrive/Current.csv', header=None)
signal_data = df[0].values.astype(float)
signal_data = signal_data - np.mean(signal_data)  # убираем постоянную составляющую

print("="*70)
print("ЛАБОРАТОРНАЯ РАБОТА №2")
print("Амплитудно-частотные спектры по току и напряжению")
print("Вариант 12")
print("="*70)

# ======================================================
# 2. ПАРАМЕТРЫ СИГНАЛА
# ======================================================
dt = 0.00015625  # 0.15625 мс
fs = 1 / dt
time = np.arange(len(signal_data)) * dt

print(f"\n📊 ПАРАМЕТРЫ ДИСКРЕТИЗАЦИИ:")
print(f"   Шаг: {dt*1000:.6f} мс")
print(f"   Частота: {fs:.2f} Гц")
print(f"   Точек: {len(signal_data)}")
print(f"   Длительность: {time[-1]:.2f} с")

# ======================================================
# 3. РАСЧЁТ СПЕКТРА
# ======================================================
N = len(signal_data)
yf = fft(signal_data)
xf = fftfreq(N, dt)[:N//2]
magnitude = (2.0/N) * np.abs(yf[:N//2])

# Ограничим до 2000 Гц (для видимости гармоник)
max_freq = 2000
max_idx = np.where(xf <= max_freq)[0]
max_idx = max_idx[-1] if len(max_idx) > 0 else len(xf)

# Находим пики в спектре
peaks, props = find_peaks(magnitude[:max_idx], height=np.max(magnitude[:max_idx]) * 0.05)
peak_freqs = xf[peaks]
peak_mags = magnitude[peaks]

# Сортируем по амплитуде
sorted_peaks = sorted(zip(peak_freqs, peak_mags), key=lambda x: x[1], reverse=True)

# Определяем основную частоту (самый большой пик, не считая 0 Гц)
fundamental_freq = sorted_peaks[0][0]
fundamental_mag = sorted_peaks[0][1]

print(f"\n📈 ОСНОВНЫЕ ЧАСТОТЫ В СПЕКТРЕ:")
print(f"   Основная частота: {fundamental_freq:.2f} Гц (амплитуда {fundamental_mag:.2f})")
print(f"   Топ-5 пиков:")
for i, (f, m) in enumerate(sorted_peaks[:5]):
    print(f"      {i+1}. {f:.2f} Гц → {m:.2f}")

# ======================================================
# 4. ПОСТРОЕНИЕ ГРАФИКОВ
# ======================================================
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Временной ряд (ток/напряжение)',
                    'Амплитудно-частотный спектр'),
    vertical_spacing=0.15
)

# Временной ряд (первые 2000 точек)
fig.add_trace(
    go.Scattergl(x=time[:2000], y=signal_data[:2000],
                 mode='lines', name='Сигнал', line=dict(width=1, color='blue')),
    row=1, col=1
)

# Спектр
fig.add_trace(
    go.Scattergl(x=xf[:max_idx], y=magnitude[:max_idx],
                 mode='lines', name='АЧХ', line=dict(color='red', width=1.5)),
    row=2, col=1
)

# Отмечаем пики на спектре
fig.add_trace(
    go.Scatter(x=peak_freqs[:10], y=peak_mags[:10],
               mode='markers', name='Пики',
               marker=dict(color='green', size=8, symbol='x')),
    row=2, col=1
)

fig.update_xaxes(title_text="Время (с)", row=1, col=1)
fig.update_yaxes(title_text="Амплитуда (усл. ед.)", row=1, col=1)
fig.update_xaxes(title_text="Частота (Гц)", row=2, col=1)
fig.update_yaxes(title_text="Амплитуда спектра", row=2, col=1)

fig.update_layout(height=800, title_text=f"Вариант 12: АЧХ сигнала (основная частота {fundamental_freq:.1f} Гц)")
fig.show()

# ======================================================
# 5. АНАЛИЗ ГАРМОНИК (относительно основной частоты)
# ======================================================
print("\n" + "="*70)
print("АНАЛИЗ ВЫСШИХ ГАРМОНИК")
print("="*70)

print(f"Основная частота: {fundamental_freq:.2f} Гц")
print(f"\nГармоники (кратные {fundamental_freq:.0f} Гц):")
print("   Гармоника | Цель (Гц) | Найдено (Гц) | Амплитуда | % от основной | Статус")
print("   " + "-"*70)

harmonics_found = []

for h in range(2, 11):
    target = h * fundamental_freq
    idx = np.argmin(np.abs(xf[:max_idx] - target))
    nearest_freq = xf[idx]
    amp = magnitude[idx]
    percent = (amp / fundamental_mag) * 100

    if amp > fundamental_mag * 0.05:  # >5% от основной
        status = "ЕСТЬ ✓"
        harmonics_found.append(h)
    else:
        status = "НЕТ"

    print(f"   {h:2d}-я     | {target:6.0f} Гц  | {nearest_freq:8.2f} Гц | {amp:8.2f}   | {percent:5.1f}%      | {status}")

# ======================================================
# 6. ВЫВОД (то, что нужно сдать)
# ======================================================
print("\n" + "="*70)
print("ВЫВОД ПО РЕЗУЛЬТАТАМ АНАЛИЗА")
print("="*70)

print(f"""
1. Объект анализа: сигнал тока/напряжения (файл Current.csv)

2. Параметры сигнала:
   - Частота дискретизации: {fs:.2f} Гц
   - Длительность записи: {time[-1]:.2f} с
   - Количество точек: {len(signal_data)}

3. Результаты спектрального анализа:
   - Основная частота сигнала: {fundamental_freq:.2f} Гц
   - Амплитуда основной гармоники: {fundamental_mag:.2f} усл. ед.

4. Высшие гармоники:
""")

if len(harmonics_found) > 0:
    print(f"   ✓ Присутствуют высшие гармоники: {', '.join([str(h) for h in harmonics_found])}-я")
    print(f"   ✗ Это свидетельствует о НЕЛИНЕЙНЫХ ИСКАЖЕНИЯХ формы сигнала.")
    print(f"   ✗ Причина: работа нелинейной нагрузки (выпрямители, ШИМ-преобразователи).")
else:
    print(f"   ✓ Высшие гармоники не обнаружены (амплитуда менее 5% от основной).")
    print(f"   ✓ Форма сигнала близка к синусоидальной.")

print(f"""
5. Заключение:
   В спектре анализируемого сигнала {'ПРИСУТСТВУЮТ' if len(harmonics_found) > 0 else 'ОТСУТСТВУЮТ'} высшие гармонические составляющие.
   {'Требуется фильтрация или анализ источника нелинейных искажений.' if len(harmonics_found) > 0 else 'Качество сигнала удовлетворительное.'}
""")
print("="*70)

# ======================================================
# 7. СОХРАНЕНИЕ ФАЙЛОВ
# ======================================================
os.makedirs("variant_12_output", exist_ok=True)

# Сохраняем график
fig.write_html("variant_12_output/acf_spectrum.html")
print("\n📁 СОХРАНЁННЫЕ ФАЙЛЫ:")
print(f"   - variant_12_output/acf_spectrum.html (интерактивный график)")

# Сохраняем спектр в CSV
spectrum_df = pd.DataFrame({
    'Частота_Гц': xf[:max_idx],
    'Амплитуда_спектра': magnitude[:max_idx]
})
spectrum_df.to_csv("variant_12_output/spectrum.csv", index=False)
print(f"   - variant_12_output/spectrum.csv (данные спектра)")

# Сохраняем исходный сигнал
df.to_csv("variant_12_output/signal.csv", index=False, header=['Значение'])
print(f"   - variant_12_output/signal.csv (исходный сигнал)")

# Сохраняем в Excel
with pd.ExcelWriter("variant_12_output/data_export.xlsx") as writer:
    df.to_excel(writer, sheet_name='Исходный_сигнал', index=False)
    spectrum_df.to_excel(writer, sheet_name='Спектр', index=False)
    pd.DataFrame({
        'Параметр': ['Основная частота (Гц)', 'Амплитуда основной', 'Наличие высших гармоник', 'Гармоники'],
        'Значение': [f"{fundamental_freq:.2f}", f"{fundamental_mag:.2f}",
                     f"{'Есть' if harmonics_found else 'Нет'}",
                     f"{', '.join([str(h) for h in harmonics_found])}" if harmonics_found else 'Отсутствуют']
    }).to_excel(writer, sheet_name='Результаты', index=False)
print(f"   - variant_12_output/data_export.xlsx (Excel с результатами)")

print("\n" + "="*70)
print("ГОТОВО! Файлы в папке 'variant_12_output'")
print("="*70)

# Показываем содержимое папки
print("\nСодержимое папки variant_12_output:")
for f in os.listdir("variant_12_output"):
    print(f"   📄 {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ЛАБОРАТОРНАЯ РАБОТА №2
Амплитудно-частотные спектры по току и напряжению
Вариант 12

📊 ПАРАМЕТРЫ ДИСКРЕТИЗАЦИИ:
   Шаг: 0.156250 мс
   Частота: 6400.00 Гц
   Точек: 60287
   Длительность: 9.42 с

📈 ОСНОВНЫЕ ЧАСТОТЫ В СПЕКТРЕ:
   Основная частота: 50.00 Гц (амплитуда 94381.95)
   Топ-5 пиков:
      1. 50.00 Гц → 94381.95



АНАЛИЗ ВЫСШИХ ГАРМОНИК
Основная частота: 50.00 Гц

Гармоники (кратные 50 Гц):
   Гармоника | Цель (Гц) | Найдено (Гц) | Амплитуда | % от основной | Статус
   ----------------------------------------------------------------------
    2-я     |    100 Гц  |   100.00 Гц |    50.35   |   0.1%      | НЕТ
    3-я     |    150 Гц  |   150.00 Гц |   659.68   |   0.7%      | НЕТ
    4-я     |    200 Гц  |   200.00 Гц |    20.36   |   0.0%      | НЕТ
    5-я     |    250 Гц  |   250.00 Гц |  1525.21   |   1.6%      | НЕТ
    6-я     |    300 Гц  |   300.00 Гц |    20.92   |   0.0%      | НЕТ
    7-я     |    350 Гц  |   350.01 Гц |   786.11   |   0.8%      | НЕТ
    8-я     |    400 Гц  |   400.01 Гц |    11.14   |   0.0%      | НЕТ
    9-я     |    450 Гц  |   450.01 Гц |   106.51   |   0.1%      | НЕТ
   10-я     |    500 Гц  |   500.01 Гц |    12.34   |   0.0%      | НЕТ

ВЫВОД ПО РЕЗУЛЬТАТАМ АНАЛИЗА

1. Объект анализа: сигнал тока/напряжения (файл Current.csv)

2. Параметры сигнала:
   - Ча